# WING Fashion — Flux 產品攝影
㩒 **Runtime → Run all**，等 ~5分鐘自動出 7 張圖。

In [ ]:
# @title 0. HuggingFace Login (必要！)

from huggingface_hub import login
import getpass

token = getpass.getpass("請輸入你的 HuggingFace Token：")
login(token)
print("✅ 登入成功！")

In [ ]:
# @title 1. Install + Download Flux
!pip install -q diffusers transformers accelerate torch

from diffusers import FluxPipeline
import torch

print("📥 Downloading Flux...")
pipe = FluxPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-dev",
    torch_dtype=torch.bfloat16
)
pipe.to("cuda")
print("✅ Flux ready!")

In [ ]:
# @title 2. Generate 7 Photos
import os, random
from IPython.display import display, Image as IPImage

os.makedirs("/content/wing_output", exist_ok=True)

PRODUCT = "a Fred Perry diamond-print shirt"
STYLE = "editorial fashion photography, cold grey color palette, soft natural lighting, cinematic composition, clean minimalist background, 1:1 aspect ratio, high end luxury look, muted tones"
NEG = "people, human, model, face, text, watermark, logo, bright flash, HDR, oversaturated, warm tones"

prompts = [
  f"{PRODUCT} hung on a minimalist metal rack, studio lighting, soft shadows, cold grey concrete wall background, {STYLE}",
  f"{PRODUCT} neatly folded on a dark grey marble surface, top-down view, soft diffused light, {STYLE}",
  f"{PRODUCT} displayed on a minimalist mannequin torso, three-quarter angle, grey seamless backdrop, {STYLE}",
  f"{PRODUCT} close up detail shot of the diamond pattern fabric texture, macro photography, soft light, grey background, {STYLE}",
  "Minimalist interior corner, white walls, grey concrete floor, soft window light casting geometric shadows, empty serene atmosphere, editorial architectural photography, " + STYLE,
  "Abstract grey gradient background with subtle texture, smooth transitions from light to dark grey, clean minimal fashion editorial backdrop, " + STYLE,
  "Empty minimalist room with natural light streaming through sheer curtains, pale grey walls, subtle shadows, calm sophisticated atmosphere, " + STYLE,
]

results = []
for idx, prompt in enumerate(prompts):
    print(f"\n🎨 Generating {idx+1}/7...")
    image = pipe(
        prompt=prompt,
        guidance_scale=3.5,
        num_inference_steps=30,
        width=1024, height=1024,
        generator=torch.Generator("cuda").manual_seed(random.randint(1,999999))
    ).images[0]
    path = f"/content/wing_output/img_{idx+1:02d}.png"
    image.save(path)
    results.append(path)
    display(IPImage(path))

print(f"\n🎉 All {len(results)} images done!")

In [ ]:
# @title 3. Download ZIP
import zipfile
from google.colab import files

zip_path = "/content/wing_photos.zip"
with zipfile.ZipFile(zip_path, 'w') as zf:
    for img in results:
        zf.write(img, os.path.basename(img))
files.download(zip_path)
print("✅ Done!")